# 06 - Training and Internal-Validation Tuning

This notebook trains all classical models. Sampling is seed-controlled, scalers fit on training only, and confidence thresholds are selected only on duplicate-group-separated internal validation to maximize decided-case macro-F1 at at least 80% coverage. The official labelled validation split is not opened for tuning.

In [ ]:
import json,sys
from pathlib import Path
cwd=Path.cwd().resolve(); REPO=next((p for p in [cwd,*cwd.parents] if (p/'implementation'/'src').exists()),None)
if REPO is None: raise RuntimeError('Start Jupyter inside the repository.')
sys.path.insert(0,str(REPO/'implementation'/'src'))
from ipcv_attire.dataset_policy import load_policy
from ipcv_attire.training import train_pipeline
TRAINING_PROFILE='smoke'  # change to full on the 32 GB desktop
FORCE_RETRAIN=False
DATA=REPO/'implementation'/'data'; BUNDLE=REPO/'implementation'/'models'/f'classical-attire-{TRAINING_PROFILE}'


## Laptop and desktop profiles

`smoke` runs the real end-to-end Fashionpedia path on a fixed small subset. `full` uses all relevant samples, float32 disk-backed feature caches, and one hard-negative-mining pass. The RTX 3080 is intentionally unused: every implemented method is classical and CPU-based.

In [ ]:
manifest_path=BUNDLE/'manifest.json'
if FORCE_RETRAIN and manifest_path.exists(): raise RuntimeError('Remove the exact ignored bundle directory manually before forcing retraining.')
if not manifest_path.exists(): train_pipeline(manifest_dir=DATA/'interim'/'manifests',fashionpedia_root=DATA/'raw'/'fashionpedia',policy=load_policy(DATA/'dataset-policy.json'),bundle_dir=BUNDLE,profile_name=TRAINING_PROFILE)
metadata=json.loads(manifest_path.read_text(encoding='utf-8')); print(json.dumps(metadata,indent=2))


## Full desktop command

```powershell
python implementation/src/train_classical_pipeline.py --profile full --bundle-dir implementation/models/classical-attire-full
```

The ignored model bundle freezes the policy, HOG detectors, recognizers and fitted scalers, uncertainty thresholds, feature configuration, environment versions, and artifact hash.

In [ ]:
import joblib
payload=joblib.load(BUNDLE/'bundle.joblib'); thresholds={target:{'low':a.low_threshold,'high':a.high_threshold} for target,a in payload['recognizers'].items()}
print(json.dumps(thresholds,indent=2)); print('Bundle:',BUNDLE)
